In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from datetime import datetime

from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
import requests
from time import sleep
import random

In [5]:
url = 'https://es.wallapop.com/app/search?category_ids=17000&latitude=40.41956&longitude=-3.69196&filters_source=quick_filters'
url = 'https://es.wallapop.com/app/search?category_ids=24200&latitude=40.41956&longitude=-3.69196&filters_source=quick_filters&distance=30000&min_sale_price=20'
url = 'https://es.wallapop.com/app/search?category_ids=24200&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest&distance=30000&condition=fair,good,as_good_as_new,new,in_box,un_opened&min_sale_price=44'

empty_url = 'https://es.wallapop.com/app/search?filters_source=search_box&keywords=q2u%20microphone&latitude=40.41956&longitude=-3.69196'
few_results_url = 'https://es.wallapop.com/app/search?filters_source=search_box&keywords=shure%20sm58&latitude=40.41956&longitude=-3.69196'

url = 'https://es.wallapop.com/app/search?category_ids=24200&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest'


---
### Explorer crawler

---

In [24]:
# set up the driver
driver = webdriver.Chrome()
driver.maximize_window()

driver.get(url)
input()
try:
    view_more = driver.find_element(By.ID, "btn-load-more")
    view_more.click
    WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button__button.walla-button__button--medium.walla-button__button--primary')))
    view_more = driver.find_element(By.CSS_SELECTOR, "walla-button__button.walla-button__button--medium.walla-button__button--primary")
    view_more.click
    
    items = driver.find_elements(By.CLASS_NAME, "ItemCardList__item")
    print(items)

    driver.execute_script("arguments[0].scrollIntoView(true);", items)
    driver.execute_script("arguments[-1].scrollIntoView(true);", items)
except NoSuchElementException as e:
    print(e)



TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x0000000104e65088 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000104e5d764 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x0000000104a6c82c cxxbridge1$string$len + 88524
3   chromedriver                        0x0000000104ab0834 cxxbridge1$string$len + 367060
4   chromedriver                        0x0000000104ae848c cxxbridge1$string$len + 595500
5   chromedriver                        0x0000000104aa5474 cxxbridge1$string$len + 321044
6   chromedriver                        0x0000000104aa60e4 cxxbridge1$string$len + 324228
7   chromedriver                        0x0000000104e2ca6c cxxbridge1$str$ptr + 1656336
8   chromedriver                        0x0000000104e314c8 cxxbridge1$str$ptr + 1675372
9   chromedriver                        0x0000000104e12950 cxxbridge1$str$ptr + 1549556
10  chromedriver                        0x0000000104e31c78 cxxbridge1$s

In [ ]:

WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".walla-button__button")))
view_more = driver.find_element(By.CSS_SELECTOR, ".walla-button__button")
view_more.click

---
### Product clasifier
---

In [ ]:
'''
It sounds like you're on the right track with normalizing the strings! Here are some additional steps you can take to further categorize and group similar book titles:

1. **Spell Correction**: Use a spell-checking library like `pyspellchecker` or `SymSpell` to correct common typos and misspellings.

2. **Remove Special Characters**: Strip out any special characters, punctuation, and numbers that are not part of the book title.

3. **Tokenization**: Break down the titles into individual words (tokens) and remove common stop words (like "and", "the", etc.).

4. **Stemming/Lemmatization**: Reduce words to their base or root form using libraries like `nltk` or `spaCy`.

5. **Fuzzy Matching**: Use fuzzy matching techniques (e.g., `fuzzywuzzy` library) to compare and group titles that are similar but not identical.

6. **Clustering**: Apply clustering algorithms (e.g., K-means, DBSCAN) to group similar titles together based on their tokenized and normalized forms.

7. **Manual Review**: After automated processing, a manual review might still be necessary to ensure accuracy, especially for edge cases.

Here's a sample Python code snippet to get you started with some of these steps:
'''


import re
from spellchecker import SpellChecker
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Sample list of book titles
titles = ["HP and the sorcerer's stone", "Harry Potter and the Sorcerer's Stone", "Harry Pótter and the Sorcerer's Stone", ...]

# Normalize titles
def normalize_title(title):
    title = title.lower().strip()
    title = re.sub(r'[^\w\s]', '', title)  # Remove special characters
    title = re.sub(r'\d+', '', title)  # Remove numbers
    return title

# Spell correction
spell = SpellChecker()
def correct_spelling(title):
    corrected_title = " ".join([spell.correction(word) for word in title.split()])
    return corrected_title

# Apply normalization and spell correction
normalized_titles = [normalize_title(title) for title in titles]
corrected_titles = [correct_spelling(title) for title in normalized_titles]

# Fuzzy matching
def group_titles(titles):
    grouped_titles = {}
    for title in titles:
        match = process.extractOne(title, grouped_titles.keys(), scorer=fuzz.token_sort_ratio)
        if match and match[1] > 80:  # Adjust threshold as needed
            grouped_titles[match[0]].append(title)
        else:
            grouped_titles[title] = [title]
    return grouped_titles

grouped_titles = group_titles(corrected_titles)

# Print grouped titles
for key, group in grouped_titles.items():
    print(f"Group: {key}")
    for title in group:
        print(f" - {title}")
